In [3]:
import numpy as np, pandas as pd
from pandas_datareader import data as web

PREDICTORS = ['unemployment', 'jobs_yoy', 'inflation_yoy', 'real_wages_yoy', 'gdp_yoy']
START = '1978'                       # UMCSENT becomes monthly 1978-01

raw = web.DataReader(['UMCSENT','PAYEMS','UNRATE','CPIAUCSL','AHETPI','GDPC1'],
                     'fred', '1947-01-01')

idx = pd.date_range(raw.index.min(), raw.index.max(), freq='MS')
yoy = lambda s, p: s.dropna().pct_change(p) * 100

# quarterly -> monthly: each quarter's value covers its own 3 months, then stops
gdp_q = yoy(raw.GDPC1, 4)
gdp_m = gdp_q.reindex(idx.union(gdp_q.index)).ffill(limit=2).reindex(idx)

d = pd.DataFrame({
    'sentiment':      raw.UMCSENT.dropna(),      # NSA; monthly from 1978-01
    'unemployment':   raw.UNRATE.dropna(),       # already a rate, keep level
    'jobs_yoy':       yoy(raw.PAYEMS, 12),
    'inflation_yoy':  yoy(raw.CPIAUCSL, 12),
    'wages_yoy':      yoy(raw.AHETPI, 12),       # kept for comparison only
    'real_wages_yoy': yoy(raw.AHETPI / raw.CPIAUCSL, 12),
    'gdp_yoy':        gdp_m,
}).reindex(idx)
d.index.name = 'date'

# quarterly aggregation -- no GDP imputation, matches Gordon's frequency
dq = d.loc[START:].resample('QS').mean()

print(d.loc[START:, ['sentiment'] + PREDICTORS].dropna().shape)   # ~(582, 6)
print(dq[['sentiment'] + PREDICTORS].dropna().shape)              # ~(194, 6)

(581, 6)
(194, 6)


/var/folders/xg/2jx4h1913bx28fptfgb7xjx00000gn/T/ipykernel_76039/3413305707.py:7: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  raw = web.DataReader(['UMCSENT','PAYEMS','UNRATE','CPIAUCSL','AHETPI','GDPC1'],


In [4]:
import numpy as np, pymc as pm, arviz as az

s  = d.loc[START:, ['sentiment'] + PREDICTORS].dropna()
y  = s.sentiment.values
X  = s[PREDICTORS].values
Xs = (X - X.mean(0)) / X.std(0)         # standardize -> betas are per 1 SD

with pm.Model() as baseline:
    alpha = pm.Normal('alpha', 85, 20)
    beta  = pm.Normal('beta', 0, 5, shape=Xs.shape[1])
    sigma = pm.HalfNormal('sigma', 10)
    pm.Normal('y', alpha + Xs @ beta, sigma, observed=y)
    idata = pm.sample(1000, tune=1000, chains=4, random_seed=1)

print(az.summary(idata))

# residual = the gap, descriptively
b = idata.posterior.beta.mean(('chain','draw')).values
resid = pd.Series(y - (idata.posterior.alpha.mean().item() + Xs @ b), index=s.index)

/opt/miniconda3/envs/pymc_env/lib/python3.11/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha, beta, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 2 seconds.


           mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
alpha    83.848  0.432  83.031   84.620      0.006    0.007    4603.0   
beta[0]  -1.971  0.527  -2.931   -0.965      0.009    0.008    3728.0   
beta[1]   0.099  0.927  -1.616    1.828      0.019    0.015    2399.0   
beta[2]  -6.225  0.712  -7.506   -4.813      0.013    0.010    3018.0   
beta[3]  -0.531  0.774  -1.971    0.949      0.015    0.011    2802.0   
beta[4]   5.637  0.788   4.172    7.123      0.015    0.012    2684.0   
sigma    10.870  0.305  10.298   11.447      0.004    0.005    5113.0   

         ess_tail  r_hat  
alpha      2994.0    1.0  
beta[0]    2945.0    1.0  
beta[1]    2733.0    1.0  
beta[2]    2870.0    1.0  
beta[3]    2892.0    1.0  
beta[4]    2599.0    1.0  
sigma      3115.0    1.0  
